In [4]:
!pip install streamlit pyngrok -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 97.3 MB/s eta 0:00:00


In [9]:
%%writefile app.py

import streamlit as st
import pandas as pd
import re
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter
import requests

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
st.set_page_config(
    page_title="Emploi Sénégal — Analyse & IA",
    page_icon="🇸🇳",
    layout="wide",
    initial_sidebar_state="expanded",
)

st.markdown("""
<style>
    .main-title {
        font-size: 2rem; font-weight: 800; color: #1a3a5c;
        text-align: center; margin-bottom: 0.2rem;
    }
    .sub-title {
        font-size: 1rem; color: #666;
        text-align: center; margin-bottom: 1.5rem;
    }
    .kpi-card {
        background: linear-gradient(135deg, #f0f4ff, #e8f0fe);
        border-radius: 12px; padding: 1rem 1.2rem;
        border-left: 4px solid #3B82F6; margin-bottom: 0.5rem;
    }
    .kpi-value { font-size: 1.6rem; font-weight: 800; color: #1a3a5c; }
    .kpi-label { font-size: 0.78rem; color: #555; margin-top: 0.1rem; }
    .chat-user {
        background: #E8F4FF; border-radius: 12px 12px 2px 12px;
        padding: 0.7rem 1rem; margin: 0.4rem 0;
        text-align: right; color: #1a3a5c; font-size: 0.95rem;
    }
    .chat-assistant {
        background: #F3F4F6; border-radius: 12px 12px 12px 2px;
        padding: 0.7rem 1rem; margin: 0.4rem 0;
        color: #222; font-size: 0.95rem;
        border-left: 3px solid #3B82F6;
    }
    .section-header {
        font-size: 1.2rem; font-weight: 700; color: #1a3a5c;
        border-bottom: 2px solid #3B82F6;
        padding-bottom: 0.3rem; margin: 1.2rem 0 0.8rem 0;
    }
    .filtre-info {
        background: #fff7ed; border-left: 4px solid #f97316;
        border-radius: 6px; padding: 0.5rem 0.8rem;
        font-size: 0.85rem; color: #7c2d12; margin-bottom: 0.8rem;
    }
</style>
""", unsafe_allow_html=True)


# ─────────────────────────────────────────────
# CHARGEMENT & NETTOYAGE
# ─────────────────────────────────────────────
@st.cache_data(show_spinner="Chargement des données…")
def charger_donnees(fichier):
    df = pd.read_excel(fichier).copy()
    df.columns = df.columns.str.strip()
    avant = len(df)
    df.drop_duplicates(subset=["Lien"], inplace=True)

    df["Compétences"]            = df["Compétences"].fillna("")
    df["Catégories compétences"] = df["Catégories compétences"].fillna("")
    df["Entreprise"]             = df["Entreprise"].fillna("Inconnu")
    df["Ville"]                  = df["Ville"].fillna("Non renseignée")

    def extraire_metier(lien):
        if not isinstance(lien, str): return "Non renseigné"
        m = re.search(r"/jobseekers/(.+?)_e_\d+", lien)
        if m:
            s = re.sub(r"[_­-]+", " ", m.group(1))
            s = re.sub(r"%[0-9a-fA-F]{2}", "", s)
            return re.sub(r"\s+", " ", s).strip().title()
        return "Non renseigné"

    df["Titre"] = df["Lien"].apply(extraire_metier)

    def split_comp(val):
        if not isinstance(val, str) or not val.strip(): return []
        return [c.strip() for c in val.split(",") if c.strip()]

    df["liste_competences"] = df["Compétences"].apply(split_comp)
    df["liste_categories"]  = df["Catégories compétences"].apply(split_comp)

    contrat_map = {
        "cdi":"CDI","cdd":"CDD","stage":"Stage",
        "freelance":"Freelance","consultant":"Consultant",
        "intérim":"Intérim","interim":"Intérim","autre":"Autre",
    }
    df["Contrat"] = df["Type de contrat"].apply(
        lambda v: contrat_map.get(str(v).strip().lower(), str(v).strip()) if isinstance(v, str) else "Autre"
    )
    df["Date"]     = pd.to_datetime(df["Date de publication"], dayfirst=True, errors="coerce")
    df["Mois"]     = df["Date"].dt.to_period("M")
    df["Année"]    = df["Date"].dt.year.astype("Int64")
    df["Mois_num"] = df["Date"].dt.month
    return df, avant - len(df)


# ─────────────────────────────────────────────
# CONTEXTE IA
# ─────────────────────────────────────────────
def construire_contexte(df):
    toutes_cats = [c for lst in df["liste_categories"] for c in lst]
    toutes_comp = [c for lst in df["liste_competences"] for c in lst if c]
    top5  = pd.Series(Counter(toutes_cats)).sort_values(ascending=False).head(5)
    top10 = pd.Series(Counter(toutes_comp)).sort_values(ascending=False).head(10)
    villes   = df["Ville"].value_counts().head(10)
    contrats = df["Contrat"].value_counts()

    df2 = df.copy()
    df2["am"] = df2["Date"].dt.to_period("M")
    evol = df2.groupby("am").size().reset_index(name="n")
    evol = evol[evol["am"] < pd.Period("2026-05","M")]
    moy  = evol["n"].mean() if len(evol) else 0
    pic  = evol.loc[evol["n"].idxmax()] if len(evol) else None
    n24  = evol[evol["am"].dt.year==2024]["n"].sum()
    n25  = evol[evol["am"].dt.year==2025]["n"].sum()
    crois = (n25-n24)/n24*100 if n24>0 else 0

    return f"""
=== ANALYSE OFFRES D'EMPLOI — SÉNÉGAL ===
Données : {len(df):,} offres (2024-2026)

TOP 5 SECTEURS :
{chr(10).join([f"  {i+1}. {s} : {v:,}" for i,(s,v) in enumerate(top5.items())])}

TOP 10 COMPÉTENCES :
{chr(10).join([f"  {i+1}. {c} : {v:,}" for i,(c,v) in enumerate(top10.items())])}

GÉOGRAPHIE (top 10 villes) :
{chr(10).join([f"  {v} : {n:,} ({n/len(df)*100:.1f}%)" for v,n in villes.items()])}

ÉVOLUTION :
  Moyenne mensuelle : {moy:.0f} offres/mois
  Mois pic : {f"{pic['am']} ({pic['n']} offres)" if pic is not None else "N/A"}
  Croissance 2024→2025 : {crois:+.1f}%

CONTRATS :
{chr(10).join([f"  {c} : {n:,} ({n/len(df)*100:.1f}%)" for c,n in contrats.items()])}

Tu es expert en marché de l'emploi au Sénégal. Réponds en français avec des chiffres précis.
""".strip()


# ─────────────────────────────────────────────
# APPEL API CLAUDE
# ─────────────────────────────────────────────
def appeler_claude(messages, contexte, api_key):
    headers = {
        "Content-Type": "application/json",
        "x-api-key": api_key,
        "anthropic-version": "2023-06-01",
    }
    payload = {
        "model": "claude-haiku-4-5-20251001",
        "max_tokens": 1024,
        "system": (
            "Tu es un assistant expert en analyse du marché de l'emploi au Sénégal. "
            "Réponds en français, de façon détaillée et structurée. "
            "Cite des chiffres précis.\n\n" + contexte
        ),
        "messages": messages,
    }
    try:
        r = requests.post(
            "https://api.anthropic.com/v1/messages",
            headers=headers, json=payload, timeout=60
        )
        if r.status_code == 200:
            return r.json()["content"][0]["text"]
        return f"❌ Erreur {r.status_code} : {r.text[:300]}"
    except Exception as e:
        return f"❌ Erreur de connexion : {str(e)}"


# ─────────────────────────────────────────────
# GRAPHIQUES
# ─────────────────────────────────────────────
def fig_secteurs(df):
    data = pd.Series(Counter([c for lst in df["liste_categories"] for c in lst])).sort_values(ascending=False).head(5)
    fig, ax = plt.subplots(figsize=(8,4))
    bars = ax.barh(data.index[::-1], data.values[::-1],
                   color=sns.color_palette("Blues_r",5), edgecolor="none", height=0.55)
    for b in bars:
        ax.text(b.get_width()+20, b.get_y()+b.get_height()/2,
                f"{int(b.get_width()):,}", va="center", fontsize=8, fontweight="bold")
    ax.set_title("KPI 1 — Top 5 Secteurs", fontweight="bold")
    ax.set_xlabel("Occurrences")
    ax.margins(x=0.2); ax.tick_params(length=0)
    ax.spines[["top","right","left"]].set_visible(False)
    plt.tight_layout(); return fig

def fig_competences(df):
    data = pd.Series(Counter([c for lst in df["liste_competences"] for c in lst if c])).sort_values(ascending=False).head(10)
    fig, ax = plt.subplots(figsize=(8,4))
    bars = ax.barh(data.index[::-1], data.values[::-1],
                   color=sns.color_palette("Oranges_r",10), edgecolor="none", height=0.55)
    for b in bars:
        ax.text(b.get_width()+10, b.get_y()+b.get_height()/2,
                f"{int(b.get_width()):,}", va="center", fontsize=8, fontweight="bold")
    ax.set_title("KPI 2 — Top 10 Compétences", fontweight="bold")
    ax.set_xlabel("Fréquence")
    ax.margins(x=0.2); ax.tick_params(length=0)
    ax.spines[["top","right","left"]].set_visible(False)
    plt.tight_layout(); return fig

def fig_geographie(df):
    villes = df["Ville"].value_counts().head(10)
    fig, ax = plt.subplots(figsize=(8,4))
    colors = sns.color_palette("Set2", len(villes))
    bars = ax.bar(villes.index, villes.values, color=colors, edgecolor="none", width=0.6)
    ax.set_yscale("log")
    for bar,(ville,val) in zip(bars, villes.items()):
        c = "#E24B4A" if "saint" in ville.lower() else "#333"
        ax.text(bar.get_x()+bar.get_width()/2, val*1.5,
                f"{val:,}\n({val/len(df)*100:.1f}%)",
                ha="center", va="bottom", fontsize=7, fontweight="bold", color=c)
    ax.set_title("KPI 3 — Géographie (log)", fontweight="bold")
    ax.set_ylabel("Offres (log)")
    ax.set_xticklabels(villes.index, rotation=35, ha="right", fontsize=8)
    ax.tick_params(length=0)
    ax.spines[["top","right","left"]].set_visible(False)
    plt.tight_layout(); return fig

def fig_contrats(df):
    contrats = df["Contrat"].value_counts()
    fig, ax = plt.subplots(figsize=(7,4))
    colors = sns.color_palette("Paired", len(contrats))
    bars = ax.bar(contrats.index, contrats.values, color=colors, edgecolor="none", width=0.6)
    for b in bars:
        val = int(b.get_height())
        ax.text(b.get_x()+b.get_width()/2, val+4,
                f"{val:,}\n({val/len(df)*100:.1f}%)",
                ha="center", va="bottom", fontsize=8, fontweight="bold")
    ax.set_title("KPI 5 — Types de Contrat", fontweight="bold")
    ax.set_ylabel("Nombre d'offres")
    ax.set_xticklabels(contrats.index, rotation=30, ha="right", fontsize=9)
    ax.tick_params(length=0); ax.margins(y=0.2)
    ax.spines[["top","right","left"]].set_visible(False)
    plt.tight_layout(); return fig

def fig_evolution(df):
    df2 = df.copy()
    df2["am"] = df2["Date"].dt.to_period("M")
    evol = df2.groupby("am").size().reset_index(name="n")
    evol = evol[evol["am"] < pd.Period("2026-05","M")].reset_index(drop=True)
    evol["label"] = evol["am"].dt.strftime("%b %y")
    evol["crois"] = evol["n"].pct_change()*100
    moy = evol["n"].mean()

    fig,(ax1,ax2) = plt.subplots(2,1,figsize=(10,7),gridspec_kw={"height_ratios":[2,1]})
    fig.suptitle("KPI 4 — Évolution Mensuelle", fontweight="bold", fontsize=13)
    x = range(len(evol)); y = evol["n"].values
    ax1.fill_between(x, y, alpha=0.12, color="#378ADD")
    ax1.plot(x, y, color="#378ADD", linewidth=2)
    ax1.scatter(x, y, color="#378ADD", s=40, edgecolor="white", linewidth=1.5, zorder=4)
    ax1.axhline(moy, color="#D85A30", linewidth=1.2, linestyle="--", alpha=0.8)
    ax1.text(len(x)-0.3, moy+5, f"Moy. {moy:.0f}", color="#D85A30", fontsize=8, ha="right")
    ax1.set_xticks(list(x))
    ax1.set_xticklabels(evol["label"], rotation=40, ha="right", fontsize=8)
    ax1.set_ylabel("Nombre d'offres"); ax1.tick_params(length=0)
    ax1.spines[["top","right"]].set_visible(False)

    crois = evol["crois"].fillna(0).values
    ax2.bar(range(len(evol)), crois,
            color=["#1D9E75" if c>=0 else "#E24B4A" for c in crois],
            width=0.65, edgecolor="none")
    ax2.axhline(0, color="#444", linewidth=0.8, alpha=0.5)
    ax2.set_xticks(range(len(evol)))
    ax2.set_xticklabels(evol["label"], rotation=40, ha="right", fontsize=8)
    ax2.set_ylabel("Croissance (%)"); ax2.tick_params(length=0)
    ax2.spines[["top","right"]].set_visible(False)
    ax2.legend(handles=[
        mpatches.Patch(color="#1D9E75", label="Hausse"),
        mpatches.Patch(color="#E24B4A", label="Baisse")
    ], fontsize=8, loc="upper left", framealpha=0.5)
    plt.tight_layout(); return fig


# ─────────────────────────────────────────────
# SIDEBAR
# ─────────────────────────────────────────────
with st.sidebar:
    st.markdown("## 🇸🇳 Emploi Sénégal")
    st.markdown("---")

    st.markdown("### 📂 Fichier de données")
    fichier = st.file_uploader("Importer le fichier Excel (.xlsx)", type=["xlsx"])

    st.markdown("---")
    st.markdown("### 🤖 Clé API Claude")
    api_key = st.text_input("Clé API", type="password", placeholder="sk-ant-...")
    st.caption("Obtenez une clé gratuite sur [console.anthropic.com](https://console.anthropic.com)")

    # Filtres — uniquement si fichier chargé
    filtre_contrat = []
    filtre_ville   = []
    filtre_annee   = []

    if fichier:
        df_raw, doublons = charger_donnees(fichier)

        st.markdown("---")
        st.markdown("### 🔍 Filtres")

        # Filtre contrat
        contrats_dispo = sorted(df_raw["Contrat"].dropna().unique().tolist())
        filtre_contrat = st.multiselect(
            "📋 Type de contrat",
            options=contrats_dispo,
            default=[],
            placeholder="Tous les contrats"
        )

        # Filtre ville
        villes_dispo = df_raw["Ville"].value_counts().head(20).index.tolist()
        filtre_ville = st.multiselect(
            "📍 Ville",
            options=villes_dispo,
            default=[],
            placeholder="Toutes les villes"
        )

        # Filtre année
        annees_dispo = sorted([
            int(a) for a in df_raw["Année"].dropna().unique()
        ])
        filtre_annee = st.multiselect(
            "📅 Année",
            options=annees_dispo,
            default=[],
            placeholder="Toutes les années"
        )

        # Compteur en temps réel
        df_preview = df_raw.copy()
        if filtre_contrat: df_preview = df_preview[df_preview["Contrat"].isin(filtre_contrat)]
        if filtre_ville:   df_preview = df_preview[df_preview["Ville"].isin(filtre_ville)]
        if filtre_annee:   df_preview = df_preview[df_preview["Année"].isin(filtre_annee)]

        st.markdown("---")
        pct = len(df_preview)/len(df_raw)*100
        st.metric("Offres sélectionnées", f"{len(df_preview):,}", f"{pct:.0f}% du total")

        if doublons > 0:
            st.caption(f"ℹ️ {doublons} doublons supprimés au chargement")

    st.markdown("---")
    st.caption("Projet de stage · Analyse emploi Sénégal 🇸🇳")


# ─────────────────────────────────────────────
# HEADER
# ─────────────────────────────────────────────
st.markdown("<div class='main-title'>📊 Analyse des Offres d'Emploi au Sénégal</div>", unsafe_allow_html=True)
st.markdown("<div class='sub-title'>Dashboard interactif + Assistant IA · Données 2024–2026</div>", unsafe_allow_html=True)

if not fichier:
    st.info("👈 Importez votre fichier Excel dans la barre latérale pour commencer.")
    st.stop()

# Application des filtres
df_filtered = df_raw.copy()
if filtre_contrat: df_filtered = df_filtered[df_filtered["Contrat"].isin(filtre_contrat)]
if filtre_ville:   df_filtered = df_filtered[df_filtered["Ville"].isin(filtre_ville)]
if filtre_annee:   df_filtered = df_filtered[df_filtered["Année"].isin(filtre_annee)]

# Bandeau filtres actifs
filtres_actifs = []
if filtre_contrat: filtres_actifs.append(f"Contrat : {', '.join(filtre_contrat)}")
if filtre_ville:   filtres_actifs.append(f"Ville : {', '.join(filtre_ville)}")
if filtre_annee:   filtres_actifs.append(f"Année : {', '.join(map(str,filtre_annee))}")
if filtres_actifs:
    st.markdown(
        f'<div class="filtre-info">🔎 Filtres actifs — {" | ".join(filtres_actifs)} — <b>{len(df_filtered):,} offres</b></div>',
        unsafe_allow_html=True
    )

tab1, tab2, tab3 = st.tabs(["📈 Dashboard KPIs", "🔎 Données brutes", "🤖 Assistant IA"])


# ─────────────────────────────────────────────
# TAB 1 — DASHBOARD
# ─────────────────────────────────────────────
with tab1:
    toutes_cats = [c for lst in df_filtered["liste_categories"] for c in lst]
    toutes_comp = [c for lst in df_filtered["liste_competences"] for c in lst if c]
    top1_sect = pd.Series(Counter(toutes_cats)).sort_values(ascending=False).index[0] if toutes_cats else "N/A"
    top1_comp = pd.Series(Counter(toutes_comp)).sort_values(ascending=False).index[0] if toutes_comp else "N/A"
    ville_top = df_filtered["Ville"].value_counts().index[0] if len(df_filtered) else "N/A"
    contrat_top = df_filtered["Contrat"].value_counts().index[0] if len(df_filtered) else "N/A"

    c1,c2,c3,c4,c5 = st.columns(5)
    c1.markdown(f'<div class="kpi-card"><div class="kpi-value">{len(df_filtered):,}</div><div class="kpi-label">Offres totales</div></div>', unsafe_allow_html=True)
    c2.markdown(f'<div class="kpi-card"><div class="kpi-value">{df_filtered["Entreprise"].nunique():,}</div><div class="kpi-label">Entreprises</div></div>', unsafe_allow_html=True)
    c3.markdown(f'<div class="kpi-card"><div class="kpi-value">{top1_sect[:13]}</div><div class="kpi-label">Secteur #1</div></div>', unsafe_allow_html=True)
    c4.markdown(f'<div class="kpi-card"><div class="kpi-value">{top1_comp[:13]}</div><div class="kpi-label">Compétence #1</div></div>', unsafe_allow_html=True)
    c5.markdown(f'<div class="kpi-card"><div class="kpi-value">{ville_top}</div><div class="kpi-label">Ville principale</div></div>', unsafe_allow_html=True)

    st.markdown("")
    col_a, col_b = st.columns(2)
    with col_a: st.pyplot(fig_secteurs(df_filtered))
    with col_b: st.pyplot(fig_competences(df_filtered))

    col_c, col_d = st.columns(2)
    with col_c: st.pyplot(fig_geographie(df_filtered))
    with col_d: st.pyplot(fig_contrats(df_filtered))

    st.pyplot(fig_evolution(df_filtered))


# ─────────────────────────────────────────────
# TAB 2 — DONNÉES BRUTES
# ─────────────────────────────────────────────
with tab2:
    st.markdown('<div class="section-header">🔎 Exploration des données</div>', unsafe_allow_html=True)

    cols_affich = [c for c in ["Titre","Entreprise","Ville","Contrat","Date","Compétences","Catégories compétences"]
                   if c in df_filtered.columns]

    recherche = st.text_input("🔍 Rechercher (titre ou compétence)", placeholder="Ex : développeur, marketing, CDI…")
    if recherche:
        mask = (df_filtered["Titre"].str.contains(recherche, case=False, na=False) |
                df_filtered["Compétences"].str.contains(recherche, case=False, na=False))
        df_show = df_filtered[mask]
    else:
        df_show = df_filtered

    st.caption(f"{len(df_show):,} offres affichées")
    st.dataframe(df_show[cols_affich].reset_index(drop=True), use_container_width=True, height=450)

    csv = df_show[cols_affich].to_csv(index=False, encoding="utf-8-sig").encode()
    st.download_button("⬇️ Télécharger CSV", csv, "offres_filtrees.csv", "text/csv")


# ─────────────────────────────────────────────
# TAB 3 — ASSISTANT IA
# ─────────────────────────────────────────────
with tab3:
    st.markdown('<div class="section-header">🤖 Assistant IA — Posez vos questions</div>', unsafe_allow_html=True)

    if not api_key:
        st.warning("⚠️ Entrez votre clé API Claude dans la barre latérale pour utiliser l'assistant.")

        st.markdown("### 🔑 Comment obtenir une clé gratuite ?")
        col1, col2 = st.columns(2)
        with col1:
            st.markdown("""
**Étapes simples :**
1. Allez sur [console.anthropic.com](https://console.anthropic.com)
2. Cliquez sur **Sign Up** → créez un compte
3. Allez dans **API Keys** → **Create Key**
4. Copiez la clé `sk-ant-...`
5. Collez-la dans la barre latérale
            """)
        with col2:
            st.info("""
💰 **Crédit offert**
Anthropic offre **$5 de crédits gratuits** à la création du compte.

Avec le modèle **Haiku** utilisé ici (le moins cher), cela représente environ **1000 à 2000 questions** gratuites.

Aucune carte bancaire nécessaire pour commencer.
            """)
        st.stop()

    # Init session
    if "messages" not in st.session_state:
        st.session_state.messages = []
    if "contexte_ia" not in st.session_state:
        st.session_state.contexte_ia = ""

    ctx = construire_contexte(df_filtered)
    if ctx != st.session_state.contexte_ia:
        st.session_state.contexte_ia = ctx

    # Questions suggérées
    st.markdown("**💡 Questions suggérées :**")
    suggestions = [
        "Quel est le profil type des offres au Sénégal ?",
        "Quelles compétences développer pour trouver un emploi ?",
        "Comment évolue le marché entre 2024 et 2026 ?",
        "Quels secteurs recrutent le plus à Dakar ?",
        "Quelle est la part des CDI versus CDD ?",
        "Quelles entreprises recrutent le plus ?",
    ]
    cols_s = st.columns(3)
    for i, q in enumerate(suggestions):
        if cols_s[i % 3].button(q, key=f"sug_{i}"):
            st.session_state._auto_q = q

    st.markdown("---")

    # Historique
    for msg in st.session_state.messages:
        css   = "chat-user" if msg["role"]=="user" else "chat-assistant"
        icone = "🧑" if msg["role"]=="user" else "🤖"
        st.markdown(f'<div class="{css}">{icone} {msg["content"]}</div>', unsafe_allow_html=True)

    # Saisie
    question = st.chat_input("Posez votre question sur le marché de l'emploi au Sénégal…")

    if hasattr(st.session_state, "_auto_q") and st.session_state._auto_q:
        question = st.session_state._auto_q
        st.session_state._auto_q = None

    if question:
        st.session_state.messages.append({"role":"user","content":question})
        st.markdown(f'<div class="chat-user">🧑 {question}</div>', unsafe_allow_html=True)
        with st.spinner("Analyse en cours…"):
            rep = appeler_claude(st.session_state.messages, st.session_state.contexte_ia, api_key)
        st.session_state.messages.append({"role":"assistant","content":rep})
        st.markdown(f'<div class="chat-assistant">🤖 {rep}</div>', unsafe_allow_html=True)
        st.rerun()

    if st.session_state.messages:
        if st.button("🗑️ Effacer la conversation"):
            st.session_state.messages = []
            st.rerun()

Overwriting app.py


In [10]:
import subprocess, time, threading
from pyngrok import ngrok

ngrok.kill()
ngrok.set_auth_token("3Cl2LqU8WGfaN68j5JTLcDH0bnz_5moAk3rmgJAziWNTXHqwv")  # ← remplacez ici

def run():
    subprocess.run([
        "streamlit", "run", "app.py",
        "--server.port=8501",
        "--server.headless=true"
    ])

t = threading.Thread(target=run, daemon=True)
t.start()
time.sleep(5)

url = ngrok.connect(8501)
print("✅ Votre app est disponible ici :")
print(url)

✅ Votre app est disponible ici :
NgrokTunnel: "https://secular-granular-endurable.ngrok-free.dev" -> "http://localhost:8501"
